# Verify ISIMIP Data Processing for Joshua Tree National Park

This notebook visualizes the processed climate data to verify that the spatial subsetting is correct.

**Processing parameters:**
- Region: Joshua Tree National Park (JOTR)
- Longitude: -116.555° to -115.445°
- Latitude: 33.655° to 34.345°
- Variable: Precipitation (pr)
- Scenario: Historical

In [ ]:
# Import required libraries
import xarray as xr
import geopandas as gpd
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from pathlib import Path
import numpy as np

In [ ]:
# Define paths
data_dir = Path('data/processed')
shapefile_path = Path('data/Joshua_Tree_National_Park.shp')

# List all processed files
processed_files = sorted(data_dir.glob('*_jotr_monthly.nc'))
print(f"Found {len(processed_files)} processed files:")
for f in processed_files:
    print(f"  - {f.name}")

In [ ]:
# Load shapefile for Joshua Tree National Park
jotr_gdf = gpd.read_file(shapefile_path)
print(f"Shapefile CRS: {jotr_gdf.crs}")
print(f"Shapefile bounds: {jotr_gdf.total_bounds}")
print(f"\nShapefile info:")
print(jotr_gdf.info())

In [ ]:
# Load one of the processed datasets to inspect
sample_file = processed_files[0]
ds = xr.open_dataset(sample_file)
print(f"\nLoaded: {sample_file.name}")
print("\nDataset structure:")
print(ds)
print(f"\nLongitude range: {float(ds.lon.min()):.3f} to {float(ds.lon.max()):.3f}")
print(f"Latitude range: {float(ds.lat.min()):.3f} to {float(ds.lat.max()):.3f}")
print(f"Time range: {ds.time.min().values} to {ds.time.max().values}")

In [ ]:
# Expected bounds from processing script
expected_bounds = {
    'lonmin': -116.555,
    'lonmax': -115.445,
    'latmin': 33.655,
    'latmax': 34.345
}

print("Verification of spatial extent:")
print(f"Expected lon range: {expected_bounds['lonmin']} to {expected_bounds['lonmax']}")
print(f"Actual lon range:   {float(ds.lon.min()):.3f} to {float(ds.lon.max()):.3f}")
print(f"Expected lat range: {expected_bounds['latmin']} to {expected_bounds['latmax']}")
print(f"Actual lat range:   {float(ds.lat.min()):.3f} to {float(ds.lat.max()):.3f}")

In [ ]:
# Plot: Data extent with JOTR shapefile overlay
fig, ax = plt.subplots(figsize=(12, 8), subplot_kw={'projection': ccrs.PlateCarree()})

# Select a sample time slice (e.g., first month)
data_slice = ds.pr.isel(time=0)

# Plot the data
data_slice.plot(ax=ax, transform=ccrs.PlateCarree(), 
                cmap='viridis', 
                cbar_kwargs={'label': 'Precipitation (kg m-2 s-1)', 'shrink': 0.8})

# Overlay the JOTR shapefile
jotr_gdf.boundary.plot(ax=ax, color='red', linewidth=2, label='JOTR Boundary')

# Add features
ax.coastlines(resolution='10m', linewidth=0.5)
ax.add_feature(cfeature.BORDERS, linewidth=0.5)
ax.add_feature(cfeature.STATES, linewidth=0.5, edgecolor='gray')

# Set extent to show the region
ax.set_extent([expected_bounds['lonmin'] - 0.2, 
               expected_bounds['lonmax'] + 0.2,
               expected_bounds['latmin'] - 0.2, 
               expected_bounds['latmax'] + 0.2], 
              crs=ccrs.PlateCarree())

ax.gridlines(draw_labels=True, linewidth=0.5, alpha=0.5)
ax.set_title(f'Processed Data Extent Check\n{sample_file.name}\n{data_slice.time.values}', fontsize=12)
ax.legend(loc='upper right')

plt.tight_layout()
plt.show()

In [ ]:
# Plot: Time series of mean precipitation over the region
fig, ax = plt.subplots(figsize=(14, 6))

# Calculate spatial mean over time
pr_mean = ds.pr.mean(dim=['lat', 'lon'])

# Convert to more readable units (mm/day)
pr_mean_mm_day = pr_mean * 86400  # Convert kg m-2 s-1 to mm/day

pr_mean_mm_day.plot(ax=ax, linewidth=1.5)
ax.set_ylabel('Mean Precipitation (mm/day)')
ax.set_xlabel('Time')
ax.set_title(f'Mean Precipitation Time Series - JOTR Region\n{sample_file.name}')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Plot: Multiple time snapshots with shapefile overlay
fig, axes = plt.subplots(2, 3, figsize=(18, 10), 
                         subplot_kw={'projection': ccrs.PlateCarree()})
axes = axes.flatten()

# Select 6 evenly spaced time steps
time_indices = np.linspace(0, len(ds.time)-1, 6, dtype=int)

for idx, time_idx in enumerate(time_indices):
    ax = axes[idx]
    data_slice = ds.pr.isel(time=time_idx)
    
    # Plot data
    im = data_slice.plot(ax=ax, transform=ccrs.PlateCarree(),
                         cmap='Blues', add_colorbar=False,
                         vmin=0, vmax=ds.pr.quantile(0.95))
    
    # Overlay shapefile
    jotr_gdf.boundary.plot(ax=ax, color='red', linewidth=2)
    
    # Add features
    ax.coastlines(resolution='10m', linewidth=0.5)
    ax.add_feature(cfeature.STATES, linewidth=0.5, edgecolor='gray')
    
    # Set extent
    ax.set_extent([expected_bounds['lonmin'] - 0.1, 
                   expected_bounds['lonmax'] + 0.1,
                   expected_bounds['latmin'] - 0.1, 
                   expected_bounds['latmax'] + 0.1], 
                  crs=ccrs.PlateCarree())
    
    ax.gridlines(draw_labels=True, linewidth=0.3, alpha=0.5, 
                 xlabel_style={'size': 8}, ylabel_style={'size': 8})
    ax.set_title(f"{data_slice.time.dt.strftime('%Y-%m').values}", fontsize=10)

# Add a single colorbar for all subplots
fig.colorbar(im, ax=axes, orientation='horizontal', 
             label='Precipitation (kg m-2 s-1)', 
             pad=0.05, shrink=0.8)

fig.suptitle(f'Precipitation Snapshots - {sample_file.name}', fontsize=14, y=0.98)
plt.tight_layout()
plt.show()

In [ ]:
# Compare all models for a specific month
fig, axes = plt.subplots(2, 3, figsize=(18, 10),
                         subplot_kw={'projection': ccrs.PlateCarree()})
axes = axes.flatten()

# Load and plot first time step from each model
for idx, file in enumerate(processed_files):
    if idx >= 6:  # Only plot first 6 models
        break
    
    ax = axes[idx]
    ds_model = xr.open_dataset(file)
    data_slice = ds_model.pr.isel(time=0)
    
    # Plot data
    im = data_slice.plot(ax=ax, transform=ccrs.PlateCarree(),
                         cmap='Blues', add_colorbar=False,
                         vmin=0)
    
    # Overlay shapefile
    jotr_gdf.boundary.plot(ax=ax, color='red', linewidth=2)
    
    # Add features
    ax.coastlines(resolution='10m', linewidth=0.5)
    ax.add_feature(cfeature.STATES, linewidth=0.5, edgecolor='gray')
    
    # Set extent
    ax.set_extent([expected_bounds['lonmin'] - 0.1, 
                   expected_bounds['lonmax'] + 0.1,
                   expected_bounds['latmin'] - 0.1, 
                   expected_bounds['latmax'] + 0.1], 
                  crs=ccrs.PlateCarree())
    
    ax.gridlines(draw_labels=True, linewidth=0.3, alpha=0.5,
                 xlabel_style={'size': 8}, ylabel_style={'size': 8})
    
    # Extract model name
    model_name = file.name.split('_')[0]
    ax.set_title(f"{model_name}\n{data_slice.time.dt.strftime('%Y-%m').values}", fontsize=10)
    
    ds_model.close()

# Hide extra subplots if fewer than 6 models
for idx in range(len(processed_files), 6):
    axes[idx].axis('off')

# Add colorbar
fig.colorbar(im, ax=axes, orientation='horizontal', 
             label='Precipitation (kg m-2 s-1)', 
             pad=0.05, shrink=0.8)

fig.suptitle('Model Comparison - First Month', fontsize=14, y=0.98)
plt.tight_layout()
plt.show()

In [ ]:
# Summary statistics
print("\n" + "="*70)
print("PROCESSING VERIFICATION SUMMARY")
print("="*70)

for file in processed_files:
    ds_check = xr.open_dataset(file)
    print(f"\n{file.name}")
    print(f"  Lon range: [{float(ds_check.lon.min()):.4f}, {float(ds_check.lon.max()):.4f}]")
    print(f"  Lat range: [{float(ds_check.lat.min()):.4f}, {float(ds_check.lat.max()):.4f}]")
    print(f"  Grid shape: {ds_check.pr.shape}")
    print(f"  Time steps: {len(ds_check.time)}")
    print(f"  Time range: {ds_check.time.min().values} to {ds_check.time.max().values}")
    print(f"  Mean precip: {float(ds_check.pr.mean()):.6f} kg m-2 s-1")
    ds_check.close()

print(f"\n" + "="*70)
print("Expected bounds (from processing script):")
print(f"  Lon: [{expected_bounds['lonmin']}, {expected_bounds['lonmax']}]")
print(f"  Lat: [{expected_bounds['latmin']}, {expected_bounds['latmax']}]")
print("="*70)

In [ ]:
# Close the dataset
ds.close()
print("\n✓ Processing verification complete!")